# SentinelFlow — 02: Fog Layer (Correlator Agent)

**Stack:** AMD ROCm · vLLM · Qwen2.5  
**Layer:** Tier 2 — fog/regional node  
**Datasets:** `ippatel03/paderborn-db` · `ipythonx/mvtec-ad`  
**Latency budget:** ≤ 200 ms per correlation window

### What this notebook does
1. Ingest edge events from `edge_events.json` (produced by Notebook 01)  
2. Group events into 5-second time windows  
3. Apply hard-coded correlation rules (fast, deterministic)  
4. **Qwen LLM reasoning** — reviews novel patterns rules don't cover  
5. Queue backpressure monitor — signals edge filter when queue is deep  
6. Escalation filter — only high/critical events go to cloud  
7. Save correlated events for Notebook 03

> **Dataset-specific rules added**  
> * Paderborn: consecutive bearing-fault detections → escalate immediately  
> * MVTec-AD: defect density > 2 per window → HIGH severity

## 0. Load config + edge events

In [ ]:
import yaml, json, pathlib
cfg = yaml.safe_load(pathlib.Path('config.yml').read_text())
VLLM_BASE_URL = cfg['vllm_base_url']
QWEN_MODEL    = 'qwen'

edge_events = json.loads(pathlib.Path('edge_events.json').read_text())
print(f'Loaded {len(edge_events)} edge events')

# Detect which dataset generated these events
DATASET_MODE = edge_events[0].get('dataset_mode', 'synthetic') if edge_events else 'synthetic'
print(f'Dataset mode: {DATASET_MODE}')

from openai import OpenAI
client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-needed')
print('Qwen client ready')

## 1. Imports

In [ ]:
import time, uuid, json
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime   import datetime, timedelta
from typing     import Dict, List, Optional, Tuple

BUDGET_WINDOW_MS   = 50
BUDGET_RULES_MS    = 20
BUDGET_QWEN_MS     = 3000
BACKPRESSURE_LIMIT = 50    # if queue depth > this → signal edge to throttle
print('✓ Imports done')

## 2. Backpressure queue monitor

In [ ]:
class BackpressureMonitor:
    """
    Monitors event queue depth.
    When depth > limit → signal edge agents to increase discard threshold.

    Production: publish a 'throttle' message to a Kafka control topic:
        producer.send('sentinelflow.control',
                      key=b'edge_throttle',
                      value=json.dumps({'action':'throttle','new_threshold':0.05}).encode())
    """
    def __init__(self, limit: int = BACKPRESSURE_LIMIT):
        self.limit    = limit
        self._history = deque(maxlen=100)
        self._throttle_events: List[Dict] = []

    def check(self, queue_depth: int) -> bool:
        """Returns True if edge agents should throttle."""
        self._history.append((datetime.utcnow().isoformat(), queue_depth))
        if queue_depth > self.limit:
            msg = (f'  ⚠ BACKPRESSURE: queue depth {queue_depth} > {self.limit} — '
                   'signalling edge to increase discard threshold')
            print(msg)
            self._throttle_events.append({
                'timestamp': datetime.utcnow().isoformat(),
                'depth': queue_depth,
                'action': 'throttle_edge',
            })
            return True
        return False

    def avg_depth(self) -> float:
        if not self._history:
            return 0.0
        return sum(d for _, d in self._history) / len(self._history)

    def throttle_summary(self) -> Dict:
        return {'total_throttle_events': len(self._throttle_events),
                'avg_queue_depth': round(self.avg_depth(), 2)}


bp = BackpressureMonitor()
print('BackpressureMonitor ready')

## 3. Time-window grouper

In [ ]:
class WindowGrouper:
    """Groups events into fixed time windows."""

    def __init__(self, window_s: float = 5.0):
        self.window_s = window_s

    def group(self, events: List[dict]) -> Dict[int, List[dict]]:
        t0 = time.perf_counter()
        if not events:
            return {}
        # Use event index as proxy for time (synthetic data has no real timestamps)
        windows: Dict[int, List] = defaultdict(list)
        for i, ev in enumerate(events):
            slot = i // max(int(len(events) / 4), 1)   # 4 windows
            windows[slot].append(ev)
        ms = (time.perf_counter() - t0) * 1000
        if ms > BUDGET_WINDOW_MS:
            print(f'  ⚠ WindowGrouper exceeded budget: {ms:.1f}ms')
        return dict(windows)


grouper = WindowGrouper()
windows = grouper.group(edge_events)
print(f'Formed {len(windows)} windows from {len(edge_events)} events')
for wid, evs in windows.items():
    labels = [e['label'] for e in evs]
    print(f'  Window {wid}: {len(evs)} events  labels={set(labels)}')

## 4. Deterministic correlation rules (dataset-aware)

In [ ]:
@dataclass
class CorrelatedEvent:
    correlation_id  : str
    event_type      : str
    severity        : str
    involved_sources: List[str]
    source_events   : List[dict]
    description     : str
    should_escalate : bool
    window_id       : int
    rule_matched    : str
    avg_confidence  : float
    dataset_mode    : str = 'synthetic'
    qwen_reasoning  : Optional[str] = None


# ── Surveillance / generic rules ─────────────────────────────────────────────
RULES_SURVEILLANCE = [
    dict(name='multi_cam_person',   labels=['person'],                    min_src=2, severity='medium',  escalate=True),
    dict(name='multi_cam_vehicle',  labels=['vehicle'],                   min_src=2, severity='high',    escalate=True),
    dict(name='hc_unknown',         labels=['unknown_object'],            min_src=1, severity='high',    escalate=True, min_conf=0.80),
    dict(name='cross_sensor_mixed', labels=['person','vehicle','ship'],   min_src=2, severity='critical',escalate=True),
    dict(name='wildfire_any',       labels=['wildfire_smoke'],            min_src=1, severity='critical',escalate=True),
]

# ── MVTec-AD specific rules ───────────────────────────────────────────────────
RULES_MVTEC = [
    dict(name='defect_cluster',     labels=['scratch','crack','hole','contamination','bent','broken'],
                                                                          min_src=1, severity='high',    escalate=True, min_count=2),
    dict(name='critical_defect',    labels=['hole','broken'],             min_src=1, severity='critical',escalate=True, min_conf=0.75),
    dict(name='surface_anomaly',    labels=['contamination','bent'],      min_src=1, severity='medium',  escalate=True),
]

# ── Paderborn bearing-fault rules ────────────────────────────────────────────
RULES_PADERBORN = [
    dict(name='bearing_fault_confirmed', labels=['bearing_fault'],        min_src=1, severity='high',    escalate=True, min_conf=0.70),
    dict(name='imbalance_detected',      labels=['imbalance'],            min_src=1, severity='medium',  escalate=True),
    dict(name='misalignment_detected',   labels=['misalignment'],         min_src=1, severity='medium',  escalate=True),
    dict(name='multi_fault',             labels=['bearing_fault','imbalance','misalignment'],
                                                                          min_src=1, severity='critical',escalate=True, min_count=2),
]

RULES_BY_MODE = {
    'synthetic' : RULES_SURVEILLANCE,
    'mvtec'     : RULES_MVTEC + RULES_SURVEILLANCE,
    'paderborn' : RULES_PADERBORN,
}


class CorrelationEngine:
    def __init__(self, dataset_mode: str = DATASET_MODE):
        self.rules = RULES_BY_MODE.get(dataset_mode, RULES_SURVEILLANCE)
        print(f'  CorrelationEngine loaded {len(self.rules)} rules for mode={dataset_mode}')

    def apply(self, window_id: int, events: List[dict]) -> List[CorrelatedEvent]:
        t0     = time.perf_counter()
        by_lbl = defaultdict(list)
        for ev in events:
            by_lbl[ev['label']].append(ev)

        found = []
        for rule in self.rules:
            matched = []
            for lbl in rule['labels']:
                sub = by_lbl.get(lbl, [])
                if 'min_conf' in rule:
                    sub = [e for e in sub if e['confidence'] >= rule['min_conf']]
                matched.extend(sub)
            if not matched:
                continue
            # min_count support (e.g. defect cluster requires ≥2 detections)
            if len(matched) < rule.get('min_count', 1):
                continue
            sources = list({e['source_id'] for e in matched})
            if len(sources) < rule.get('min_src', 1):
                continue
            avg_c = sum(e['confidence'] for e in matched) / len(matched)
            found.append(CorrelatedEvent(
                correlation_id   = str(uuid.uuid4()),
                event_type       = rule['name'],
                severity         = rule['severity'],
                involved_sources = sources,
                source_events    = matched,
                description      = f"{rule['name']} across {len(sources)} source(s) ({len(matched)} detections)",
                should_escalate  = rule['escalate'],
                window_id        = window_id,
                rule_matched     = rule['name'],
                avg_confidence   = round(avg_c, 4),
                dataset_mode     = DATASET_MODE,
            ))

        ms = (time.perf_counter() - t0) * 1000
        if ms > BUDGET_RULES_MS:
            print(f'  ⚠ CorrelationEngine exceeded budget: {ms:.1f}ms')
        return found


engine = CorrelationEngine()
print('CorrelationEngine ready')

## 5. Qwen LLM reasoning — novel pattern detection

In [ ]:
# Dataset-aware system prompts for Qwen fog reasoning
_FOG_SYSTEM_BASE = """\
You are the Fog Correlator Agent for SentinelFlow.
Events from a time window did NOT match any hard-coded rule.
Your job: identify novel patterns, assign severity, decide whether to escalate.

Respond ONLY with valid JSON:
{
  "pattern_detected": true/false,
  "pattern_name": "<short name or null>",
  "severity": "low|medium|high|critical",
  "escalate": true/false,
  "reasoning": "<2 sentences>",
  "involved_labels": ["<label>", ...]
}
"""

_FOG_SYSTEM_MVTEC = _FOG_SYSTEM_BASE + """
CONTEXT: Events come from MVTec-AD industrial texture inspection.
Labels are defect types: scratch, crack, hole, contamination, bent, broken.
Consider spatial co-occurrence of defect types as an additional signal.
"""

_FOG_SYSTEM_PADERBORN = _FOG_SYSTEM_BASE + """
CONTEXT: Events come from Paderborn bearing-fault vibration signals.
Labels are fault classes: bearing_fault, imbalance, misalignment.
A combination of fault types in a short window often indicates compound mechanical failure.
"""

FOG_SYSTEMS = {
    'synthetic' : _FOG_SYSTEM_BASE,
    'mvtec'     : _FOG_SYSTEM_MVTEC,
    'paderborn' : _FOG_SYSTEM_PADERBORN,
}


class QwenFogReasoner:
    def __init__(self, model: str = QWEN_MODEL, dataset_mode: str = DATASET_MODE):
        self.model  = model
        self.system = FOG_SYSTEMS.get(dataset_mode, _FOG_SYSTEM_BASE)

    def reason(self, window_id: int,
               events: List[dict],
               already_matched: List[str]) -> Optional[Dict]:
        """Ask Qwen to find novel patterns not caught by hard-coded rules."""
        summary = [{
            'label'     : e['label'],
            'confidence': e['confidence'],
            'source_id' : e['source_id'],
            'motion'    : e.get('motion_score', 0),
            'dataset'   : e.get('dataset_mode', 'unknown'),
        } for e in events]

        user_msg = (
            f'Window {window_id}. Events: {json.dumps(summary)}\n'
            f'Rules already matched: {already_matched}\n'
            'Identify any additional pattern not covered by existing rules.'
        )

        t0 = time.perf_counter()
        try:
            resp = client.chat.completions.create(
                model=self.model,
                messages=[
                    {'role': 'system', 'content': self.system},
                    {'role': 'user',   'content': user_msg},
                ],
                max_tokens=256,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            if raw.startswith('```'):
                raw = '\n'.join(l for l in raw.split('\n')
                                if not l.strip().startswith('```')).strip()
            result = json.loads(raw)
        except Exception as e:
            result = None
            print(f'  QwenFog parse error: {e}')

        ms = (time.perf_counter() - t0) * 1000
        if ms > BUDGET_QWEN_MS:
            print(f'  ⚠ QwenFog exceeded budget: {ms:.0f}ms')
        return result


fog_reasoner = QwenFogReasoner()
print('QwenFogReasoner ready')

## 6. Full fog pipeline — process all windows

In [ ]:
all_correlated: List[CorrelatedEvent] = []

print(f'Processing {len(windows)} windows  (dataset_mode={DATASET_MODE})...\n')

for wid, w_events in windows.items():
    print(f'── Window {wid} ({len(w_events)} events) ─────────────────────────')

    # Backpressure check
    bp.check(len(w_events))

    # Deterministic rules
    rule_matches = engine.apply(wid, w_events)
    matched_names = [c.rule_matched for c in rule_matches]
    for c in rule_matches:
        sev_icon = {'low':'🔵','medium':'🟡','high':'🟠','critical':'🔴'}.get(c.severity,'⚪')
        print(f'  {sev_icon} RULE: {c.event_type}  severity={c.severity}  '
              f'sources={c.involved_sources}  conf={c.avg_confidence:.3f}')

    # Qwen reasoning for novel patterns
    qwen_result = fog_reasoner.reason(wid, w_events, matched_names)
    if qwen_result and qwen_result.get('pattern_detected'):
        sev  = qwen_result.get('severity', 'low')
        esc  = qwen_result.get('escalate', False)
        name = qwen_result.get('pattern_name', 'novel_pattern')
        print(f'  🤖 QWEN: {name}  severity={sev}  escalate={esc}')
        print(f'     → {qwen_result.get("reasoning", "")}')

        if rule_matches:
            rule_matches[0].qwen_reasoning = qwen_result.get('reasoning')
        else:
            all_correlated.append(CorrelatedEvent(
                correlation_id   = str(uuid.uuid4()),
                event_type       = name or 'novel_pattern',
                severity         = sev,
                involved_sources = list({e['source_id'] for e in w_events}),
                source_events    = w_events,
                description      = qwen_result.get('reasoning', ''),
                should_escalate  = esc,
                window_id        = wid,
                rule_matched     = 'qwen_novel',
                avg_confidence   = sum(e['confidence'] for e in w_events)/len(w_events),
                dataset_mode     = DATASET_MODE,
                qwen_reasoning   = qwen_result.get('reasoning'),
            ))
    elif qwen_result:
        print(f'  🤖 QWEN: no novel pattern detected')

    all_correlated.extend(rule_matches)
    print()

escalate = [c for c in all_correlated if c.should_escalate]
retain   = [c for c in all_correlated if not c.should_escalate]
print(f'Total correlated : {len(all_correlated)}')
print(f'Escalate → cloud : {len(escalate)}')
print(f'Retain locally   : {len(retain)}')
print(f'\nBackpressure summary: {bp.throttle_summary()}')

## 7. Save escalated events for Notebook 03

In [ ]:
import pathlib

def correlated_to_dict(c: CorrelatedEvent) -> dict:
    return c.__dict__.copy()

payload = [correlated_to_dict(c) for c in escalate]
pathlib.Path('correlated_events.json').write_text(
    json.dumps(payload, default=str, indent=2)
)
print(f'Saved {len(payload)} escalated events → correlated_events.json')
print(f'\nBackpressure avg queue depth: {bp.avg_depth():.1f}')